<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c34-header.png" alt="Nextia Learning · Anomaly Detection: Find What Is Unusual" width="100%">

# Drift in many features, and what to do about it

**[Drift in many features, and what to do about it](https://learning.nextia-ai.com/courses/anomaly-detection/m03/drift-in-many-features/)** · Anomaly Detection: Find What Is Unusual · Module 3, lesson 6 of 7 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** check many features every day without drowning in false alarms: the Bonferroni correction, false-discovery-rate control (Benjamini-Hochberg) and ranking drift by its size. You build a per-feature drift report by hand.

| | |
|---|---|
| **Time** | About 35 minutes |
| **Needs** | An internet connection for the setup cells (about 7 MB). No account. |
| **You should know** | KS, chi-square and PSI, from [Has the distribution changed?](https://learning.nextia-ai.com/courses/anomaly-detection/m03/has-the-distribution-changed/). |
| **You do not need** | A drift tool: the report is about 30 lines of pandas and SciPy. |
| **Tested** | Python 3.12.13 in Jupyter on macOS (Apple silicon), from a fresh start (*Restart and run all*), with the package versions of the setup cell |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/anomaly-detection/m03/drift-in-many-features/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C34/notebooks/M03-L06-drift-in-many-features/drift-in-many-features-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs the versions that the course uses: pandas 3.1.1, numpy 2.5.4, scipy 1.18.1, scikit-learn 1.9.1, matplotlib 3.11.2. It needs an internet connection. The first time, it takes up to a minute.

You should see one line per package, with the same versions. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again. A restart is needed when an older version was already loaded.

In [ ]:
%pip install -q --disable-pip-version-check pandas==3.1.1 numpy==2.5.4 scipy==1.18.1 scikit-learn==1.9.1 matplotlib==3.11.2
from importlib.metadata import version
for p in ['pandas', 'numpy', 'scipy', 'scikit-learn', 'matplotlib']:
    print(p, version(p))

## Setup: get the course data

Run the next cell. It downloads the data that this lesson uses (`payments.csv.gz`, `sealed-key.txt`) into a folder `anomaly-course/data`, and checks each file's checksum. It also downloads `compare.py`, the course's shared code. It needs an internet connection and no account. It takes a few seconds.

You should see `Ready:` and the files with their sizes. If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `wrong checksum`: delete the folder `anomaly-course` (in Colab: the **Files** panel on the left), then run the cell again.

The data is made up for this course: no real people, cards or payments. [The dataset card](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C34/data/dataset-card.md) describes every table.

In [ ]:
# Setup: download the course data into anomaly-course/data/ and check each file.
# Standard library only.
import hashlib, os, shutil, subprocess, urllib.request
from pathlib import Path

LABS = "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C34/"
FILES = {
    "payments.csv.gz": "c4e18c1bbaf422cfc99821f2a1b65cf0e8274f835c40ca757a37a531eeccd542",
    "sealed-key.txt": "144cfd16737bdfcc4e992ad8ede0b8856265a2193c89fa7266fe0b27cb62ff98",
}
CODE_FILES = ["compare.py"]
HERE = Path.cwd()
# The notebook works in its own folder anomaly-course/, marked with a file, so it never writes into another project.
if Path(".nextia-notebook").exists():          # the cell ran before: start again from the parent folder
    os.chdir("..")
    HERE = Path.cwd()
WORK = Path("anomaly-course").resolve()
(WORK / "data").mkdir(parents=True, exist_ok=True)
(WORK / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

def fetch(name, url, path, expected=None):
    """Use a copy next to the notebook if there is one, else download it."""
    if path.exists() and (expected is None or sha256(path) == expected):
        return
    if (HERE / name).exists():
        shutil.copy(HERE / name, path)
    else:
        download(url, path)
    if expected and sha256(path) != expected:
        raise SystemExit(f"{name} has the wrong checksum. Delete the folder anomaly-course and run again.")

for name, expected in FILES.items():
    fetch(name, LABS + "data/" + name, WORK / "data" / name, expected)
for name in CODE_FILES:
    fetch(name, LABS + name, WORK / name)
os.chdir(WORK)
print("Ready:", ", ".join(f"{n} ({(Path('data') / n).stat().st_size / 1e6:.1f} MB)" for n in FILES))

## Setup: helpers

The next cell defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.

It also loads pandas as `pd`, NumPy as `np` and matplotlib as `plt`. Run the cell. It prints nothing.

In [ ]:
# Helpers: check an answer without showing it, and the chart style.
import hashlib, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
pd.set_option("display.precision", 3)
pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 20)
warnings.filterwarnings("ignore", category=FutureWarning)
plt.rcParams.update({"figure.figsize": (9, 3.2), "axes.spines.top": False, "axes.spines.right": False})

def fingerprint(value):
    def norm(v):
        if hasattr(v, "to_dict") and hasattr(v, "columns"):
            return {"columns": [str(c) for c in v.columns], "rows": [norm(list(r)) for r in v.itertuples(index=False)]}
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple, set)):
            return [norm(x) for x in (sorted(v, key=str) if isinstance(v, set) else v)]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 4)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. Ten p-values on one day

A test with a limit of 0.05 says "changed" by chance in 1 case of 20 when nothing changed. Ten features a day give a false alarm about every second day.

- The **Bonferroni correction** divides the limit by the number of tests: 0.05 / 10 = 0.005. It almost never gives a false alarm, and it misses many real changes.
- The **Benjamini-Hochberg (BH) procedure** controls the **false discovery rate (FDR)**: among the tests that it keeps, at most 5% are false on average. Sort the p-values, give each a limit of 0.05 × rank / 10, and keep every test up to the largest rank that is under its limit.

> **Predict.** How many of these ten tests does each method keep?

> **Check.** You should see 5 below 0.05, 1 after Bonferroni and 3 after BH.

In [ ]:
p = pd.Series([0.001, 0.008, 0.012, 0.030, 0.041, 0.120, 0.250, 0.380, 0.600, 0.900], index=list("ABCDEFGHIJ"))
m = len(p)                                                  # ten features checked today
tiny = pd.DataFrame({"p": p, "rank": p.rank().astype(int)})
tiny["p < 0.05"] = tiny.p < 0.05
tiny["Bonferroni: p < 0.05 / 10"] = tiny.p < 0.05 / m
tiny["BH limit: 0.05 x rank / 10"] = 0.05 * tiny["rank"] / m
largest = tiny["rank"][tiny.p <= tiny["BH limit: 0.05 x rank / 10"]].max()   # the largest rank under its limit
tiny["BH"] = tiny["rank"] <= largest
print(tiny.to_string())

Run the next cell. It defines `check()` (one test and the PSI for one feature), `psi()` and `bh()`. It prints nothing.

In [ ]:
from scipy import stats
NUMBERS = ["amount", "hour"]
CATEGORIES = ["device_type", "merchant_category", "account_type", "region", "country", "currency"]
FEATURES = NUMBERS + CATEGORIES

def psi(ref, cur, bins=10):
    """PSI of a number, with 10 bins from the quantiles of the reference."""
    edges = np.unique(np.quantile(ref, np.linspace(0, 1, bins + 1)))
    edges[0], edges[-1] = -np.inf, np.inf
    r = np.clip(np.histogram(ref, edges)[0] / len(ref), 1e-6, None)
    c = np.clip(np.histogram(cur, edges)[0] / len(cur), 1e-6, None)
    return float(np.sum((c - r) * np.log(c / r)))

def check(ref, cur, feature):
    """The test and the size of the change of one feature: (test, p-value, PSI)."""
    if feature in NUMBERS:
        return "KS", float(stats.ks_2samp(ref[feature], cur[feature]).pvalue), psi(ref[feature].to_numpy(), cur[feature].to_numpy())
    cats = sorted(set(ref[feature]) | set(cur[feature]))
    r, c = ref[feature].value_counts().reindex(cats, fill_value=0), cur[feature].value_counts().reindex(cats, fill_value=0)
    p_value = float(stats.chi2_contingency(np.vstack([r, c])).pvalue)
    rs, cs = np.clip(r / r.sum(), 1e-6, None), np.clip(c / c.sum(), 1e-6, None)
    return "chi-square", p_value, float(np.sum((cs - rs) * np.log(cs / rs)))

def bh(pvalues, q=0.05):
    """Benjamini-Hochberg: True for each test kept as a discovery, at a false discovery rate q."""
    p = np.asarray(pvalues)
    order = np.argsort(p)
    passed = p[order] <= q * np.arange(1, len(p) + 1) / len(p)
    keep = np.zeros(len(p), bool)
    if passed.any():
        keep[order[: np.flatnonzero(passed).max() + 1]] = True
    return keep

In [ ]:
payments = pd.read_csv("data/payments.csv.gz", parse_dates=["ts"])
payments["day"] = payments.ts.dt.floor("D")
july = payments[(payments.day >= "2025-07-01") & (payments.day <= "2025-07-31")]
august = payments[(payments.day >= "2025-08-01") & (payments.day <= "2025-08-31")]
print(f"July {len(july):,} payments, August {len(august):,}")

## 2. A month with nothing new

Each day of August, we check the eight payment features against July: 31 × 8 = 248 tests.

> **Predict.** If nothing changed, about 5% of 248, so 12, would be below 0.05 by chance. How many will you see?

> **Check.** You should see 62 below 0.05, 33 after Bonferroni and 45 after BH.

In [ ]:
rows = [(str(d.date()), f, check(july, g, f)[1]) for d, g in august.groupby("day") for f in FEATURES]
stable = pd.DataFrame(rows, columns=["day", "feature", "p"])
print(f"{len(stable)} tests: p < 0.05 in {(stable.p < 0.05).sum()}, "
      f"Bonferroni {(stable.p < 0.05 / len(stable)).sum()}, FDR (BH) {bh(stable.p).sum()}")
print(stable[stable.p < 0.05].feature.value_counts().to_string())

Far more than 12. Most are on `country` and `currency`. A burst of planted fraud from abroad really moves these two features on the day it happens. So not all of these alarms are "false": a test cannot tell a fraud burst from a change of normal. BH keeps 45; Bonferroni keeps 33.

## 3. A drift report by hand

Now the window after the new region (2025-09-15 to 2025-10-14) against August. For each feature: the test, its p-value, BH, and the size of the change (PSI, small below 0.1, large above 0.25). The report is sorted by size.

In [ ]:
after = payments[(payments.day >= "2025-09-15") & (payments.day <= "2025-10-14")]
report = pd.DataFrame([(f, *check(august, after, f)) for f in FEATURES], columns=["feature", "test", "p", "PSI"])
report["BH"] = bh(report.p)
report["size"] = pd.cut(report.PSI, [-1, 0.1, 0.25, np.inf], labels=["small", "moderate", "large"])
report = report.sort_values("PSI", ascending=False)
print(report.to_string(index=False, formatters={"p": "{:.3g}".format, "PSI": "{:.4f}".format}))

> **Check.** You should see `region` first, with a large PSI.

Every feature has a tiny p-value: with about 55,000 payments in each window, every change is "significant". Only one change is large: the region mix. Ranked by size, the report tells you where to look first.

## 4. Every day for six weeks

Run the next cell. It runs the report every day from 2025-09-01 to 2025-10-14, against August. It takes a few seconds.

In [ ]:
window = payments[(payments.day >= "2025-09-01") & (payments.day <= "2025-10-14")]
rows = [(d, f, *check(august, g, f)) for d, g in window.groupby("day") for f in FEATURES]
daily = pd.DataFrame(rows, columns=["day", "feature", "test", "p", "PSI"])
daily["BH"] = bh(daily.p)
print(f"{daily.day.nunique()} days x {len(FEATURES)} features = {len(daily)} checks")
print(f"p < 0.05: {(daily.p < 0.05).sum()}; FDR (BH): {daily.BH.sum()}; PSI 0.1 or more: {(daily.PSI >= 0.1).sum()}")
print(daily[daily.PSI >= 0.1].feature.value_counts().to_string())

> **Check.** You should see 191 checks kept by BH and 56 with a PSI of 0.1 or more.

By p-value, 191 of 352 checks say "changed". By size, 56 do, and most are the region: the planned new region, which you can write down once and stop alarming on.

## 5. Drift is a reason to look

The country feature passed a PSI of 0.1 on a few days. What happened on those days? The next cell uses the course's answer key (`compare.planted_payments()`). The key is packed in `data/sealed-key.txt`, and you can unpack it: it is there to score your own work, not to keep a secret.

In [ ]:
import compare
planted = compare.planted_payments()          # unpacks the answer key: you see the answers from here on
window = window.assign(planted=window.payment_id.isin(planted.payment_id), abroad=window.country != "CA")
country_days = set(daily[(daily.feature == "country") & (daily.PSI >= 0.1)].day)
per_day = window.groupby("day").apply(lambda g: pd.Series({"payments abroad": g.abroad.sum(),
                                                         "planted payments abroad": (g.abroad & g.planted).sum()}))
per_day["country PSI 0.1 or more"] = per_day.index.isin(country_days)
print(per_day.groupby("country PSI 0.1 or more").mean().round(1).to_string())

On the days with country drift, there were 48.8 planted payments abroad on average, against 13.9 on the other days. The drift check found the days of the fraud bursts. It did not find which payments: that is the job of a detector. Drift is a reason to look, not a proof of a problem.

## 6. Check yourself

**Your turn 1.** Use BH on the ten p-values of section 1 with q = 0.10 instead of 0.05. How many tests does it keep? Write the number as `bh_10`.

In [ ]:
bh_10 = ...   # your code

In [ ]:
expect('bh_10', bh_10, 5)

**Your turn 2.** In the 248 August tests (`stable`), how many are below 0.05 on a feature other than `country` and `currency`? Compare your number with the 12 that chance alone gives. Write it as `other_features`.

In [ ]:
other_features = ...   # your code

In [ ]:
expect('other_features', other_features, 22)

**Your turn 3.** In the report of section 3, which feature has the second-largest PSI? Write its name as a string in `second_feature`.

In [ ]:
second_feature = ...   # your code

In [ ]:
expect_hash('second_feature', second_feature, '9e93fa75ae43215b')

## Recap

- Many tests a day give false alarms by chance. Bonferroni divides the limit by the number of tests; BH controls the share of false discoveries.
- With big samples, every feature is "significant". Rank drift by its size: in six weeks, 191 checks by p-value, 56 by PSI of 0.1 or more.
- A drift alarm is a reason to look. Here it led to the days of planted fraud bursts, and to a planned new region.

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Benford's law and other business checks](https://learning.nextia-ai.com/courses/anomaly-detection/m03/benfords-law-and-other-business-checks/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/anomaly-detection/m03/drift-in-many-features/)

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The practice data is synthetic and has no real people.